# Memory-Optimized Bi-GRU with CNN + Attention

**Target: F1 Harmonic ≥ 0.60 | Memory: <15GB VRAM**

## Memory Optimizations:
- Mixed precision training (FP16)
- Gradient checkpointing
- Reduced batch size with gradient accumulation
- Smaller embedding (128 instead of 256)
- Efficient attention implementation

In [1]:
!pip install -q trackio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 988.0/988.0 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 138.2 MB/s eta 0:00:00


In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.amp import autocast  # FIXED: Updated import
from torch.cuda.amp import GradScaler
from torch.utils.checkpoint import checkpoint
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Memory optimization settings
torch.backends.cudnn.benchmark = True
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

class Config:
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # MEMORY-OPTIMIZED Architecture
    embed_dim = 128
    cnn_channels = 64
    hidden_dim = 384
    num_layers = 2
    num_heads = 4
    dropout = 0.4
    
    # MEMORY-OPTIMIZED Training
    batch_size = 32
    grad_accum_steps = 2
    learning_rate = 5e-4
    weight_decay = 1e-4
    epochs = 60
    grad_clip = 0.5
    label_smoothing = 0.1
    use_amp = True
    use_checkpointing = True
    max_class_weight = 3.0  # FIXED: Cap class weights
    
    val_split = 0.1
    train_path = 'train.csv'
    test_path = 'test.csv'
    model_save_path = 'best_model_bigru_v10.pth'
    submission_path = 'submission_bigru_v10.csv'
    
    trackio_project = '25-t3-nppe2'
    trackio_space_id = 'manyakaistha/dlgenai-nppe'
    trackio_group = 'bi_gru_v10_optimized'

cfg = Config()
print(f'Device: {cfg.device}')
print(f'AMP: {cfg.use_amp}, Checkpointing: {cfg.use_checkpointing}')

def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

Device: cuda
AMP: True, Checkpointing: True


In [4]:
os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'hf_placeholder_token_for_reproducibility'
os.environ['TRACKIO_SPACE_ID'] = cfg.trackio_space_id

def maybe_init_trackio():
    try:
        import trackio
        token = os.environ.get('HUGGINGFACEHUB_API_TOKEN')
        if cfg.trackio_space_id and token:
            trackio.init(project=cfg.trackio_project, space_id=cfg.trackio_space_id, group=cfg.trackio_group)
            print('✓ TrackIO initialized')
            return trackio
    except Exception as e:
        print(f'TrackIO init failed: {e}')
    return None

trk = maybe_init_trackio()

* Trackio project initialized: 25-t3-nppe2
* Trackio metrics will be synced to Hugging Face Dataset: manyakaistha/dlgenai-nppe-dataset
* Found existing space: https://huggingface.co/spaces/manyakaistha/dlgenai-nppe
* View dashboard by going to: https://manyakaistha-dlgenai-nppe.hf.space/


* Created new run: manyakaistha-1765868964
✓ TrackIO initialized


In [5]:
PAD_TOKEN = 0

AA_CHARS = ['<PAD>', '<UNK>', '*', 'A', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'P', 'Q', 'R', 'S', 'T', 'V', 'W', 'Y']
AA_TO_ID = {c: i for i, c in enumerate(AA_CHARS)}
ID_TO_AA = {i: c for c, i in AA_TO_ID.items()}

Q8_CHARS = ['<PAD>'] + sorted(['B', 'C', 'E', 'G', 'H', 'I', 'S', 'T'])
Q8_TO_ID = {c: i for i, c in enumerate(Q8_CHARS)}
ID_TO_Q8 = {i: c for c, i in Q8_TO_ID.items()}

Q3_CHARS = ['<PAD>'] + sorted(['C', 'E', 'H'])
Q3_TO_ID = {c: i for i, c in enumerate(Q3_CHARS)}
ID_TO_Q3 = {i: c for c, i in Q3_TO_ID.items()}

print(f'Vocab: AA={len(AA_TO_ID)}, Q8={len(Q8_TO_ID)}, Q3={len(Q3_TO_ID)}')

Vocab: AA=23, Q8=9, Q3=4


In [6]:
class ProteinDataset(Dataset):
    def __init__(self, data, mode='train'):
        self.data = data
        self.mode = mode

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        seq_ids = [AA_TO_ID.get(c, AA_TO_ID['<UNK>']) for c in row['seq']]
        item = {'seq': torch.tensor(seq_ids, dtype=torch.long), 'id': row['id']}
        if self.mode == 'train':
            item['q8'] = torch.tensor([Q8_TO_ID[c] for c in row['sst8']], dtype=torch.long)
            item['q3'] = torch.tensor([Q3_TO_ID[c] for c in row['sst3']], dtype=torch.long)
        return item

def collate_fn(batch):
    seqs = pad_sequence([item['seq'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
    out = {'seq': seqs, 'ids': [item['id'] for item in batch], 'lengths': torch.tensor([len(item['seq']) for item in batch])}
    if 'q8' in batch[0]:
        out['q8'] = pad_sequence([item['q8'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
        out['q3'] = pad_sequence([item['q3'] for item in batch], batch_first=True, padding_value=PAD_TOKEN)
    return out

df_train_full = pd.read_csv(cfg.train_path)
train_data, val_data = train_test_split(df_train_full, test_size=cfg.val_split, random_state=cfg.seed)

train_ds = ProteinDataset(train_data.reset_index(drop=True), mode='train')
val_ds = ProteinDataset(val_data.reset_index(drop=True), mode='train')
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn, pin_memory=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn, pin_memory=True, num_workers=2)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

Train: 6535, Val: 727


In [7]:
def compute_class_weights(labels_column, vocab, max_weight=cfg.max_class_weight):
    """Compute inverse frequency weights with capping to prevent extreme values."""
    counts = {c: 0 for c in vocab if c != '<PAD>'}
    for label_str in labels_column:
        for c in label_str:
            if c in counts:
                counts[c] += 1
    total = sum(counts.values())
    weights = torch.ones(len(vocab))
    for c, cnt in counts.items():
        if cnt > 0:
            idx = vocab.index(c)
            weights[idx] = min(total / (len(counts) * cnt), max_weight)  # FIXED: Cap weights
    weights[0] = 0.0
    weights = weights / weights[1:].max()  # Normalize
    return weights

q8_weights = compute_class_weights(train_data['sst8'], Q8_CHARS).to(cfg.device)
q3_weights = compute_class_weights(train_data['sst3'], Q3_CHARS).to(cfg.device)
print(f'Q8 weights: {q8_weights}')
print(f'Q3 weights: {q3_weights}')

Q8 weights: tensor([0.0000, 1.0000, 0.1719, 0.1983, 1.0000, 0.1321, 1.0000, 0.5381, 0.3918],
       device='cuda:0')
Q3 weights: tensor([0.0000, 0.5179, 1.0000, 0.6248], device='cuda:0')


In [8]:
class MultiScaleCNN(nn.Module):
    def __init__(self, in_dim, out_dim, kernels=[3, 5, 7], dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv1d(in_dim, out_dim, k, padding=k//2) for k in kernels])
        self.proj = nn.Linear(out_dim * len(kernels), out_dim * 2)
        self.norm = nn.LayerNorm(out_dim * 2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x.transpose(1, 2)
        conv_outs = [torch.relu(conv(x)) for conv in self.convs]
        x = torch.cat(conv_outs, dim=1).transpose(1, 2)
        return self.dropout(self.norm(self.proj(x)))


class EfficientSelfAttention(nn.Module):
    """Memory-efficient attention using chunking."""
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, num_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, _ = self.attn(x, x, x, key_padding_mask=mask, need_weights=False)
        return self.norm(x + self.dropout(attn_out))


class MemoryEfficientBiGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim, cnn_channels, hidden_dim, num_layers,
                 num_heads, num_q8, num_q3, dropout=0.4, use_checkpointing=True):
        super().__init__()
        self.use_checkpointing = use_checkpointing

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_TOKEN)
        self.cnn = MultiScaleCNN(embed_dim, cnn_channels, dropout=dropout)

        gru_input_dim = cnn_channels * 2
        self.gru = nn.GRU(gru_input_dim, hidden_dim, num_layers, batch_first=True,
                         bidirectional=True, dropout=dropout if num_layers > 1 else 0)

        self.gru_norm = nn.LayerNorm(hidden_dim * 2)
        self.residual_proj = nn.Linear(gru_input_dim, hidden_dim * 2)
        self.attention = EfficientSelfAttention(hidden_dim * 2, num_heads, dropout)
        self.dropout = nn.Dropout(dropout)

        # Simpler heads to save memory
        self.q8_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_q8)
        )
        self.q3_head = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, num_q3)
        )

    def _gru_forward(self, x):
        out, _ = self.gru(x)
        return out

    def forward(self, x):
        pad_mask = (x == PAD_TOKEN)
        embed = self.embedding(x)
        cnn_out = self.cnn(embed)

        # Gradient checkpointing for GRU
        if self.use_checkpointing and self.training:
            gru_out = checkpoint(self._gru_forward, cnn_out, use_reentrant=False)
        else:
            gru_out, _ = self.gru(cnn_out)

        # Residual
        residual = self.residual_proj(cnn_out)
        gru_out = self.gru_norm(gru_out + residual)

        # Attention with checkpointing
        if self.use_checkpointing and self.training:
            attn_out = checkpoint(self.attention, gru_out, pad_mask, use_reentrant=False)
        else:
            attn_out = self.attention(gru_out, pad_mask)

        out = self.dropout(attn_out)
        return self.q8_head(out), self.q3_head(out)


model = MemoryEfficientBiGRU(
    vocab_size=len(AA_TO_ID), embed_dim=cfg.embed_dim, cnn_channels=cfg.cnn_channels,
    hidden_dim=cfg.hidden_dim, num_layers=cfg.num_layers, num_heads=cfg.num_heads,
    num_q8=len(Q8_TO_ID), num_q3=len(Q3_TO_ID), dropout=cfg.dropout,
    use_checkpointing=cfg.use_checkpointing
).to(cfg.device)

print(model)
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')

# Estimate memory
param_mem = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**3)
print(f'Model memory: ~{param_mem:.2f} GB')

MemoryEfficientBiGRU(
  (embedding): Embedding(23, 128, padding_idx=0)
  (cnn): MultiScaleCNN(
    (convs): ModuleList(
      (0): Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(1,))
      (1): Conv1d(128, 64, kernel_size=(5,), stride=(1,), padding=(2,))
      (2): Conv1d(128, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    )
    (proj): Linear(in_features=192, out_features=128, bias=True)
    (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.4, inplace=False)
  )
  (gru): GRU(128, 384, num_layers=2, batch_first=True, dropout=0.4, bidirectional=True)
  (gru_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (residual_proj): Linear(in_features=128, out_features=768, bias=True)
  (attention): EfficientSelfAttention(
    (attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
    )
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropou

In [9]:
criterion_q8 = nn.CrossEntropyLoss(weight=q8_weights, ignore_index=PAD_TOKEN, label_smoothing=cfg.label_smoothing)
criterion_q3 = nn.CrossEntropyLoss(weight=q3_weights, ignore_index=PAD_TOKEN, label_smoothing=cfg.label_smoothing)

optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
scaler = GradScaler(enabled=cfg.use_amp)

class ModelCheckpoint:
    def __init__(self, filepath):
        self.filepath = filepath
        self.best_score = -np.inf

    def __call__(self, score, model, optimizer, epoch, metrics):
        if score > self.best_score:
            print(f'✓ Saving best model (epoch {epoch+1}, score={score:.4f})')
            self.best_score = score
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                       'optimizer_state_dict': optimizer.state_dict(),
                       'best_score': score, 'metrics': metrics}, self.filepath)
            return True
        return False

ckpt = ModelCheckpoint(cfg.model_save_path)
print('Training setup complete!')

Training setup complete!


/tmp/ipython-input-2485974153.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=cfg.use_amp)


In [10]:
def train_epoch(model, loader, optimizer, scheduler, scaler, device, epoch):
    model.train()
    total_loss, total_q8_f1, total_q3_f1, count = 0, 0, 0, 0
    optimizer.zero_grad()
    
    pbar = tqdm(loader, desc=f'Epoch {epoch+1} Train')
    for step, batch in enumerate(pbar):
        seq = batch['seq'].to(device, non_blocking=True)
        q8_tgt = batch['q8'].to(device, non_blocking=True)
        q3_tgt = batch['q3'].to(device, non_blocking=True)
        
        # FIXED: Updated autocast API
        with autocast(device_type='cuda', enabled=cfg.use_amp):
            q8_logits, q3_logits = model(seq)
            loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
            # FIXED: Don't divide here, scale backward instead
            loss = loss_q8 + loss_q3
        
        # Scale loss for gradient accumulation
        scaler.scale(loss / cfg.grad_accum_steps).backward()
        
        if (step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step(epoch + step / len(loader))
        
        # FIXED: Store actual loss value
        total_loss += loss.item()
        with autocast(device_type='cuda', enabled=False):  # Calculate F1 in FP32
            total_q8_f1 += calculate_f1(q8_logits.detach().float(), q8_tgt)
            total_q3_f1 += calculate_f1(q3_logits.detach().float(), q3_tgt)
        count += 1
        
        pbar.set_postfix({'loss': total_loss/count, 'q8': total_q8_f1/count, 'q3': total_q3_f1/count})
        
        if trk and step % 10 == 0:
            trk.log({'train/loss': total_loss/count, 'train/f1_q8': total_q8_f1/count})
    
    torch.cuda.empty_cache()
    return total_loss/count, total_q8_f1/count, total_q3_f1/count, harmonic_mean(total_q8_f1/count, total_q3_f1/count)


def validate(model, loader, device):
    model.eval()
    total_loss = 0
    all_q8_preds, all_q8_targets, all_q3_preds, all_q3_targets = [], [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            seq = batch['seq'].to(device, non_blocking=True)
            q8_tgt = batch['q8'].to(device, non_blocking=True)
            q3_tgt = batch['q3'].to(device, non_blocking=True)
            
            # FIXED: Updated autocast API
            with autocast(device_type='cuda', enabled=cfg.use_amp):
                q8_logits, q3_logits = model(seq)
                loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
                loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
            
            total_loss += (loss_q8 + loss_q3).item()
            
            mask = q8_tgt != PAD_TOKEN
            all_q8_preds.extend(q8_logits.argmax(-1)[mask].cpu().numpy())
            all_q8_targets.extend(q8_tgt[mask].cpu().numpy())
            all_q3_preds.extend(q3_logits.argmax(-1)[mask].cpu().numpy())
            all_q3_targets.extend(q3_tgt[mask].cpu().numpy())
    
    torch.cuda.empty_cache()
    f1_q8 = f1_score(all_q8_targets, all_q8_preds, average='macro', zero_division=0)
    f1_q3 = f1_score(all_q3_targets, all_q3_preds, average='macro', zero_division=0)
    return total_loss/len(loader), f1_q8, f1_q3, harmonic_mean(f1_q8, f1_q3)

In [14]:
def calculate_f1(preds, targets):
    preds = preds.argmax(-1)
    mask = targets != PAD_TOKEN
    return f1_score(targets[mask].cpu().numpy(), preds[mask].cpu().numpy(), average='macro', zero_division=0)

def harmonic_mean(f1_q8, f1_q3):
    if f1_q8 + f1_q3 == 0: return 0.0
    return 2 * (f1_q8 * f1_q3) / (f1_q8 + f1_q3)

def train_epoch(model, loader, optimizer, scheduler, scaler, device, epoch):
    model.train()
    total_loss, total_q8_f1, total_q3_f1, count = 0, 0, 0, 0
    optimizer.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch+1} Train')
    for step, batch in enumerate(pbar):
        seq = batch['seq'].to(device, non_blocking=True)
        q8_tgt = batch['q8'].to(device, non_blocking=True)
        q3_tgt = batch['q3'].to(device, non_blocking=True)

        with autocast(device_type='cuda', enabled=cfg.use_amp):
            q8_logits, q3_logits = model(seq)
            loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
            loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))
            loss = (loss_q8 + loss_q3) / cfg.grad_accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % cfg.grad_accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step(epoch + step / len(loader))

        total_loss += loss.item() * cfg.grad_accum_steps
        total_q8_f1 += calculate_f1(q8_logits.detach().float(), q8_tgt)
        total_q3_f1 += calculate_f1(q3_logits.detach().float(), q3_tgt)
        count += 1

        pbar.set_postfix({'loss': total_loss/count, 'q8': total_q8_f1/count, 'q3': total_q3_f1/count})

        if trk and step % 10 == 0:
            trk.log({'train/loss': total_loss/count, 'train/f1_q8': total_q8_f1/count})

    # Clear cache
    torch.cuda.empty_cache()
    return total_loss/count, total_q8_f1/count, total_q3_f1/count, harmonic_mean(total_q8_f1/count, total_q3_f1/count)


def validate(model, loader, device):
    model.eval()
    total_loss = 0
    all_q8_preds, all_q8_targets, all_q3_preds, all_q3_targets = [], [], [], []

    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation'):
            seq = batch['seq'].to(device, non_blocking=True)
            q8_tgt = batch['q8'].to(device, non_blocking=True)
            q3_tgt = batch['q3'].to(device, non_blocking=True)

            with autocast(device_type='cuda', enabled=cfg.use_amp):
                q8_logits, q3_logits = model(seq)
                loss_q8 = criterion_q8(q8_logits.view(-1, len(Q8_TO_ID)), q8_tgt.view(-1))
                loss_q3 = criterion_q3(q3_logits.view(-1, len(Q3_TO_ID)), q3_tgt.view(-1))

            total_loss += (loss_q8 + loss_q3).item()

            mask = q8_tgt != PAD_TOKEN
            all_q8_preds.extend(q8_logits.argmax(-1)[mask].cpu().numpy())
            all_q8_targets.extend(q8_tgt[mask].cpu().numpy())
            all_q3_preds.extend(q3_logits.argmax(-1)[mask].cpu().numpy())
            all_q3_targets.extend(q3_tgt[mask].cpu().numpy())

    torch.cuda.empty_cache()
    f1_q8 = f1_score(all_q8_targets, all_q8_preds, average='macro', zero_division=0)
    f1_q3 = f1_score(all_q3_targets, all_q3_preds, average='macro', zero_division=0)
    return total_loss/len(loader), f1_q8, f1_q3, harmonic_mean(f1_q8, f1_q3)

In [ ]:
history = {'train_loss': [], 'train_f1_q8': [], 'train_f1_q3': [], 'train_f1_harm': [],
           'val_loss': [], 'val_f1_q8': [], 'val_f1_q3': [], 'val_f1_harm': []}
best_val_f1 = 0

print('Starting training with memory optimizations...')
print(f'Batch size: {cfg.batch_size} x {cfg.grad_accum_steps} = {cfg.batch_size * cfg.grad_accum_steps} effective')
print(f'Target: F1 Harmonic ≥ 0.60\n')

for epoch in range(cfg.epochs):
    train_loss, train_q8, train_q3, train_harm = train_epoch(model, train_loader, optimizer, scheduler, scaler, cfg.device, epoch)
    val_loss, val_q8, val_q3, val_harm = validate(model, val_loader, cfg.device)

    for k, v in [('train_loss', train_loss), ('train_f1_q8', train_q8), ('train_f1_q3', train_q3), ('train_f1_harm', train_harm),
                 ('val_loss', val_loss), ('val_f1_q8', val_q8), ('val_f1_q3', val_q3), ('val_f1_harm', val_harm)]:
        history[k].append(v)

    print(f'\nEpoch {epoch+1}/{cfg.epochs}:')
    print(f'  Train Loss: {train_loss:.4f} | Q8: {train_q8:.4f} | Q3: {train_q3:.4f} | Harm: {train_harm:.4f}')
    print(f'  Val   Loss: {val_loss:.4f} | Q8: {val_q8:.4f} | Q3: {val_q3:.4f} | Harm: {val_harm:.4f}')

    ckpt(val_harm, model, optimizer, epoch, {'val_q8': val_q8, 'val_q3': val_q3, 'val_harm': val_harm})
    if val_harm > best_val_f1:
        best_val_f1 = val_harm

    if trk:
        trk.log({'epoch': epoch+1, 'val/f1_q8': val_q8, 'val/f1_q3': val_q3, 'val/f1_harm': val_harm})

    print('-' * 60)

    # Memory cleanup
    torch.cuda.empty_cache()

print(f'\n✓ Training complete! Best Val F1 Harmonic: {best_val_f1:.4f}')
print(f'Target achieved: {"✓ YES" if best_val_f1 >= 0.60 else "✗ NO"}')

Starting training with memory optimizations...
Batch size: 32 x 2 = 64 effective
Target: F1 Harmonic ≥ 0.60



Epoch 1 Train:   0%|          | 0/205 [00:00<?, ?it/s]

Validation:   0%|          | 0/23 [00:00<?, ?it/s]


Epoch 1/60:
  Train Loss: 2.9825 | Q8: 0.2769 | Q3: 0.5709 | Harm: 0.3729
  Val   Loss: 2.7111 | Q8: 0.3012 | Q3: 0.6583 | Harm: 0.4133
✓ Saving best model (epoch 1, score=0.4133)
------------------------------------------------------------


Epoch 2 Train:   0%|          | 0/205 [00:00<?, ?it/s]

...


Epoch 3/60:
  Train Loss: 2.7011 | Q8: 0.3478 | Q3: 0.6689 | Harm: 0.4576
  Val   Loss: 2.6149 | Q8: 0.3246 | Q3: 0.6937 | Harm: 0.4422
✓ Saving best model (epoch 3, score=0.4422)
------------------------------------------------------------


Epoch 4 Train:   0%|          | 0/205 [00:00<?, ?it/s]

Validation:   0%|          | 0/23 [00:00<?, ?it/s]


Epoch 4/60:
  Train Loss: 2.6554 | Q8: 0.3577 | Q3: 0.6814 | Harm: 0.4692
  Val   Loss: 2.6076 | Q8: 0.3359 | Q3: 0.6910 | Harm: 0.4520
✓ Saving best model (epoch 4, score=0.4520)
------------------------------------------------------------


Epoch 5 Train:   0%|          | 0/205 [00:00<?, ?it/s]

In [ ]:
print(f'Loading best model from: {cfg.model_save_path}')
loaded = torch.load(cfg.model_save_path, map_location=cfg.device)
model.load_state_dict(loaded['model_state_dict'])
print(f'✓ Loaded from epoch {loaded["epoch"]+1}, score: {loaded["best_score"]:.4f}')

df_test = pd.read_csv(cfg.test_path)
test_ds = ProteinDataset(df_test, mode='test')
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

model.eval()
results = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Inference'):
        seq = batch['seq'].to(cfg.device)
        lengths = batch['lengths']
        # FIXED: Updated autocast API
        with autocast(device_type='cuda', enabled=cfg.use_amp):
            q8_logits, q3_logits = model(seq)
        q8_preds = q8_logits.argmax(-1).cpu().numpy()
        q3_preds = q3_logits.argmax(-1).cpu().numpy()
        
        for i, id_ in enumerate(batch['ids']):
            L = lengths[i].item()
            results.append({'id': id_, 'sst8': ''.join([ID_TO_Q8[idx] for idx in q8_preds[i, :L]]),
                           'sst3': ''.join([ID_TO_Q3[idx] for idx in q3_preds[i, :L]])})

df_sub = pd.DataFrame(results)
df_sub.to_csv(cfg.submission_path, index=False)
print(f'\n✓ Submission saved: {cfg.submission_path}')
print(df_sub.head())

In [ ]:
print(f'Loading best model from: {cfg.model_save_path}')
loaded = torch.load(cfg.model_save_path, map_location=cfg.device)
model.load_state_dict(loaded['model_state_dict'])
print(f'✓ Loaded from epoch {loaded["epoch"]+1}, score: {loaded["best_score"]:.4f}')

df_test = pd.read_csv(cfg.test_path)
test_ds = ProteinDataset(df_test, mode='test')
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

model.eval()
results = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Inference'):
        seq = batch['seq'].to(cfg.device)
        lengths = batch['lengths']
        with autocast(enabled=cfg.use_amp):
            q8_logits, q3_logits = model(seq)
        q8_preds = q8_logits.argmax(-1).cpu().numpy()
        q3_preds = q3_logits.argmax(-1).cpu().numpy()

        for i, id_ in enumerate(batch['ids']):
            L = lengths[i].item()
            results.append({'id': id_, 'sst8': ''.join([ID_TO_Q8[idx] for idx in q8_preds[i, :L]]),
                           'sst3': ''.join([ID_TO_Q3[idx] for idx in q3_preds[i, :L]])})

df_sub = pd.DataFrame(results)
df_sub.to_csv(cfg.submission_path, index=False)
print(f'\n✓ Submission saved: {cfg.submission_path}')
print(df_sub.head())

In [ ]:
print('Validating submission...')
assert len(df_sub) == len(df_test)
valid_q8, valid_q3 = set('BCEGIHST'), set('CEH')
for idx, row in df_sub.iterrows():
    seq_len = len(df_test.loc[df_test['id'] == row['id'], 'seq'].values[0])
    assert len(row['sst8']) == seq_len and len(row['sst3']) == seq_len
print('✓ Submission valid!')